<a href="https://colab.research.google.com/github/Keistkmiya/Tugas2-MachineLearning/blob/main/Tugas2_Chapter8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Chapter 8: Linear Regression

## Setup and Library Imports


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

url_ev = "https://data.wa.gov/api/views/f6w7-q2d2/rows.csv?accessType=DOWNLOAD"
df_ev = pd.read_csv(url_ev)
df_ev.columns = [col.lower().replace(' ', '_') for col in df_ev.columns]

target_features = ['model_year', 'base_msrp']
available_features = [col for col in target_features if col in df_ev.columns]
all_needed_cols = available_features + ['electric_range']

df_reg = df_ev[all_needed_cols].dropna()
df_reg = df_reg[df_reg['electric_range'] > 0]

X = df_reg[available_features]
y = df_reg['electric_range']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Setup Chapter 8 Berhasil!")
print(f"Fitur yang digunakan: {available_features}")
print(f"Dataset siap dengan {len(df_reg)} baris data.")

Setup Chapter 8 Berhasil!
Fitur yang digunakan: ['model_year']
Dataset siap dengan 101283 baris data.


## Fitting an Ordinary Least Squares (OLS) Model

Teknik paling dasar dalam regresi adalah **Ordinary Least Squares (OLS)**. OLS bekerja dengan cara meminimalkan jumlah kuadrat dari selisih antara nilai yang diprediksi oleh garis linier dan nilai asli dalam data (Residuals).

Setelah model dilatih, kita akan memeriksa **Coefficients** untuk melihat fitur mana yang paling berpengaruh terhadap jarak tempuh mobil listrik.

In [3]:
lr = LinearRegression()
lr.fit(X_train, y_train)

print(f"Intersep (Bias): {lr.intercept_:.2f}")

for feature, coef in zip(X.columns, lr.coef_):
    print(f"Koefisien untuk {feature}: {coef:.4f}")

y_pred = lr.predict(X_test)
print(f"\nContoh Prediksi: {y_pred[0]:.2f} mil")
print(f"Nilai Asli      : {y_test.iloc[0]} mil")

Intersep (Bias): 14890.44
Koefisien untuk model_year: -7.3197

Contoh Prediksi: 82.73 mil
Nilai Asli      : 38.0 mil


## Handling Interaction Effects and Polynomial Features

Model linier standar mengasumsikan bahwa setiap fitur bersifat independen. Namun, sering kali ada **Interaction Effects**, di mana dampak satu fitur terhadap target bergantung pada nilai fitur lainnya. Kita bisa merepresentasikan ini dengan mengalikan dua fitur tersebut:
$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \beta_3 (x_1 \times x_2) + \epsilon$$

Selain itu, jika hubungan antar data tidak lurus melainkan melengkung, kita bisa menggunakan **Polynomial Features** (seperti $x^2$). Scikit-learn menyediakan `PolynomialFeatures` untuk secara otomatis membuat kombinasi fitur-fitur ini, sehingga model linier kita bisa menangkap pola yang lebih kompleks tanpa harus berganti ke model non-linier.

In [4]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

lr_poly = LinearRegression()
lr_poly.fit(X_train_poly, y_train)

print(f"Jumlah fitur awal: {X_train.shape[1]}")
print(f"Jumlah fitur setelah transformasi polinomial: {X_train_poly.shape[1]}")
print(f"Nama fitur baru: {poly.get_feature_names_out(available_features)}")

y_poly_pred = lr_poly.predict(X_test_poly)
print(f"\nPrediksi dengan Polinomial: {y_poly_pred[0]:.2f} mil")
print(f"Nilai Asli                : {y_test.iloc[0]} mil")

Jumlah fitur awal: 1
Jumlah fitur setelah transformasi polinomial: 2
Nama fitur baru: ['model_year' 'model_year^2']

Prediksi dengan Polinomial: 80.58 mil
Nilai Asli                : 38.0 mil


## Reducing Variance with Regularization (Ridge and Lasso)

Ketika kita menambah banyak fitur (seperti hasil polinomial tadi), model cenderung memiliki variansi yang tinggi dan mudah *overfitting*. **Regularization** menambahkan penalti pada ukuran koefisien agar tidak terlalu besar.

Ada dua jenis utama:
1.  **Ridge Regression (L2):** Menambahkan penalti kuadrat dari koefisien ke fungsi biaya. Rumusnya: $$\text{Cost} = \text{RSS} + \alpha \sum_{j=1}^{p} \beta_j^2$$
    Ridge cenderung mengecilkan semua koefisien secara merata tetapi tidak membuatnya menjadi nol.
2.  **Lasso Regression (L1):** Menambahkan penalti nilai absolut dari koefisien. Rumusnya: $$\text{Cost} = \text{RSS} + \alpha \sum_{j=1}^{p} |\beta_j|$$
    Lasso sangat unik karena bisa memaksa koefisien fitur yang tidak penting menjadi **nol**, sehingga berfungsi juga sebagai *feature selection*.

Nilai $\alpha$ (Alpha) adalah tuas kendali kita; semakin besar Alpha, semakin kuat "rem" yang diberikan pada model.

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_test_scaled = scaler.transform(X_test_poly)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

ridge_score = ridge.score(X_test_scaled, y_test)
lasso_score = lasso.score(X_test_scaled, y_test)

ridge_zeros = np.sum(ridge.coef_ == 0)
lasso_zeros = np.sum(lasso.coef_ == 0)

print(f"Jumlah fitur setelah Polinomial: {X_train_poly.shape[1]}")
print(f"Fitur yang dinolkan oleh Lasso: {lasso_zeros}")
print("-" * 30)
print(f"Skor R2 Ridge: {ridge_score:.4f}")
print(f"Skor R2 Lasso: {lasso_score:.4f}")

Jumlah fitur setelah Polinomial: 2
Fitur yang dinolkan oleh Lasso: 1
------------------------------
Skor R2 Ridge: 0.0849
Skor R2 Lasso: 0.0702


## Handling Outliers with Weighted Regression

Dalam *Ordinary Least Squares* (OLS) standar, setiap titik data dianggap memiliki tingkat kepentingan yang sama. Namun, di dunia nyata, terkadang ada beberapa data yang lebih "terpercaya" atau lebih relevan daripada yang lain.

**Weighted Least Squares (WLS)** memungkinkan kita memberikan bobot ($w_i$) pada setiap observasi. Model akan berusaha meminimalkan fungsi biaya yang sudah dimodifikasi:
$$\text{Cost} = \sum_{i=1}^{n} w_i (y_i - \hat{y}_i)^2$$

Dengan memberikan bobot yang berbeda, kita bisa memaksa model untuk lebih "mendengarkan" data tertentu (misalnya data terbaru) dan sedikit mengabaikan data yang kita anggap kurang akurat atau merupakan *outlier*.

In [7]:
weights = np.where(X_train['model_year'] > 2022, 3.0, 1.0)

weighted_lr = LinearRegression()
weighted_lr.fit(X_train, y_train, sample_weight=weights)

print("Perbandingan Koefisien:")
for feat, coef_std, coef_weight in zip(available_features, lr.coef_, weighted_lr.coef_):
    print(f"Fitur: {feat}")
    print(f"  - Koefisien Standar : {coef_std:.4f}")
    print(f"  - Koefisien Berbobot: {coef_weight:.4f}\n")

weighted_score = weighted_lr.score(X_test, y_test)
print(f"Skor R2 (Weighted Model): {weighted_score:.4f}")

Perbandingan Koefisien:
Fitur: model_year
  - Koefisien Standar : -7.3197
  - Koefisien Berbobot: -10.3226

Skor R2 (Weighted Model): 0.0483
